In [1]:
# Import library yang diperlukan
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras_tuner import RandomSearch
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Precision, Recall
import tensorflow as tf

In [2]:
# Konfigurasi
DATASET_PATH = r'D:\1_arsip negara\1_arsip_unsr!_on going\Semester 7\1_AICI Batch 7\Tugas Akhir\Final_Project\bisindo'
# Direktori masing-masing
train_dir = os.path.join(DATASET_PATH, 'Train')
val_dir = os.path.join(DATASET_PATH, 'Validation')
test_dir = os.path.join(DATASET_PATH, 'Test')

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 26

In [3]:
# Function untuk membuat data generator sesuai img_size dan batch_size
def get_data_generators(img_size, batch_size):
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=30,
        width_shift_range=0.15,
        height_shift_range=0.15,
        zoom_range=0.2,
        brightness_range=[0.5, 1.5],
        shear_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    val_datagen = ImageDataGenerator(
        rescale=1./255
    )
    
    train_gen = train_datagen.flow_from_directory(
        train_dir,
        target_size=(img_size, img_size),
        batch_size=batch_size,
        class_mode='categorical'
    )
    val_gen = val_datagen.flow_from_directory(
        val_dir,
        target_size=(img_size, img_size),
        batch_size=batch_size,
        class_mode='categorical'
    )
    return train_gen, val_gen


In [ ]:
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam, SGD

# Jumlah kelas harus didefinisikan
NUM_CLASSES = 26  # Ganti sesuai kebutuhan

def build_model(hp):
    img_size = hp.Choice('img_size', [112, 320])

    base_model = MobileNetV2(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(hp.Float('dropout', 0.2, 0.5, step=0.1))(x)
    outputs = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=outputs)

    # Pilih optimizer
    optimizer_choice = hp.Choice('optimizer', ['adam', 'sgd'])
    if optimizer_choice == 'adam':
        optimizer = Adam(learning_rate=hp.Choice('learning_rate', [0.001, 0.0005]))
    else:
        optimizer = SGD(learning_rate=hp.Choice('learning_rate', [0.01, 0.005]), momentum=0.9)

    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


In [9]:
from kerastuner import HyperParameters
from kerastuner.tuners import RandomSearch

# Define HyperParameters
hp = HyperParameters()
hp.Choice('batch_size', [64, 128])
hp.Choice('img_size', [112 , 320])

class CustomTuner(RandomSearch):
    def run_trial(self, trial, *args, **kwargs):
        hp = trial.hyperparameters
        img_size = hp.get('img_size')
        batch_size = hp.get('batch_size')

        train_gen, val_gen = get_data_generators(img_size, batch_size)

        model = self.hypermodel.build(hp)

        history = model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=kwargs.get('epochs', 5),
            verbose=1
        )

        # Return metrics secara eksplisit!
        return {
            "val_accuracy": history.history["val_accuracy"][-1],
            "val_loss": history.history["val_loss"][-1]
        }
        
# Setup Custom Tuner
tuner = CustomTuner(
    build_model,
    objective='val_accuracy',
    max_trials=50,
    executions_per_trial=1,
    directory=r'D:\mobilenet_random_search',
    project_name='gesture_classification_2',
    hyperparameters=hp
)

# Jalankan tuning
tuner.search(epochs=5)


Trial 10 Complete [00h 19m 57s]
val_accuracy: 0.5792540907859802

Best val_accuracy So Far: 0.7062937021255493
Total elapsed time: 02h 39m 57s

Search: Running Trial #11

Value             |Best Value So Far |Hyperparameter
64                |64                |batch_size
320               |320               |img_size
0.3               |0.4               |dropout
adam              |adam              |optimizer
0.001             |0.001             |learning_rate

Found 8036 images belonging to 26 classes.
Found 1716 images belonging to 26 classes.
Epoch 1/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 234s 2s/step - accuracy: 0.3347 - loss: 2.5736 - val_accuracy: 0.5880 - val_loss: 1.5975
Epoch 2/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 229s 2s/step - accuracy: 0.8007 - loss: 0.9729 - val_accuracy: 0.6538 - val_loss: 1.2874
Epoch 3/5
126/126 ━━━━━━━━━━━━━━━━━━━━ 294s 2s/step - accuracy: 0.8480 - loss: 0.6895 - val_accuracy: 0.6684 - val_loss: 1.1728
Epoch 4/5
 32/126 ━━━━━━━━━━━━━━━━━━━━ 3:25 2s/step - accuracy:

KeyboardInterrupt: 

In [ ]:
# Optimalisasi Hyperparameter dengan Random Search

tuner = RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=28,
    executions_per_trial=1,
    directory='tuning_mobilenetv2',
    project_name='mobilenet_random_search'
)

tuner.search(
    train_generator,
    validation_data=val_generator,
    epochs=4
)


Trial 28 Complete [00h 07m 04s]
val_accuracy: 0.808857798576355

Best val_accuracy So Far: 0.8216783404350281
Total elapsed time: 03h 22m 05s


In [ ]:
# Hasil Parameter Tuning terbaik
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best Hyperparameters:")
print("Learning rate:", best_hps.get('learning_rate'))
print("Dense units:", best_hps.get('dense_units'))
print("Dropout:", best_hps.get('dropout'))


Best Hyperparameters:
Learning rate: 0.001
Dense units: 256
Dropout: 0.30000000000000004


In [20]:
import pandas as pd

# Ambil semua trial dari tuner
trials = tuner.oracle.trials

# Buat list hasil tuning
results = []
for trial_id, trial in trials.items():
    row = trial.hyperparameters.values.copy()  # ambil kombinasi hyperparam
    row['trial_id'] = trial_id
    row['val_accuracy'] = trial.score          # ambil val_accuracy
    results.append(row)

# Buat DataFrame
df_results = pd.DataFrame(results)

# Urutkan berdasarkan val_accuracy tertinggi
df_results = df_results.sort_values(by='val_accuracy', ascending=False)

# Tampilkan sebagai tabel
print("=== Tabel Hasil Tuning ===")
print(df_results)


=== Tabel Hasil Tuning ===
    dropout  dense_units  learning_rate trial_id  val_accuracy
0       0.3          256         0.0010       00      0.821678
11      0.2          128         0.0010       11      0.812937
27      0.3          192         0.0010       27      0.808858
24      0.4          128         0.0010       24      0.799534
4       0.3          128         0.0010       04      0.798368
5       0.4          256         0.0010       05      0.797786
6       0.2          256         0.0010       06      0.795455
16      0.2           64         0.0010       16      0.772727
12      0.4           64         0.0010       12      0.768065
19      0.3           64         0.0010       19      0.750583
14      0.2          256         0.0001       14      0.704545
8       0.4          256         0.0001       08      0.698718
17      0.3          128         0.0100       17      0.689977
23      0.3          192         0.0001       23      0.681235
25      0.2          192    

In [ ]:
# Simpan ke dalam file CSV
df_results.to_csv("hasil_tuning_mobilenetv2.csv", index=False)

print("Hasil tuning telah disimpan ke dalam file 'hasil_tuning - mobilenetv2.csv'")